# Part 1: Linear Regression for Stellar Luminosity
## Modeling L as a function of M using Linear Regression

In this notebook, we implement **linear regression from first principles** to model the relationship between stellar mass (M) and luminosity (L):

$$\hat{L} = w \cdot M + b$$

We will:
1. Visualize the dataset
2. Implement the MSE loss function
3. Visualize the cost surface
4. Derive and implement gradients
5. Implement gradient descent (non-vectorized and vectorized)
6. Analyze convergence and experiment with learning rates
7. Interpret results and discuss limitations

## 1. Import Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Configure matplotlib for inline plots
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)

# Define the dataset
# Part I: One feature (Mass vs Luminosity)
M = np.array([0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.4])  # stellar mass (solar masses)
L = np.array([0.15, 0.35, 1.00, 2.30, 4.10, 7.00, 11.2, 17.5, 25.0, 35.0])  # stellar luminosity (solar luminosities)
n_samples = len(M)

print(f"Dataset: {n_samples} stellar samples")
print(f"Mass range: {M.min():.1f} - {M.max():.1f} M☉")
print(f"Luminosity range: {L.min():.2f} - {L.max():.1f} L☉")

## 2. Dataset Visualization

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
ax1.scatter(M, L, s=100, alpha=0.7, edgecolors='black', linewidth=1.5)
ax1.set_xlabel('Mass (M☉)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Luminosity (L☉)', fontsize=12, fontweight='bold')
ax1.set_title('Stellar Mass vs Luminosity (Linear Scale)', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Log scale
ax2.loglog(M, L, 'o', markersize=8, alpha=0.7)
ax2.set_xlabel('Mass (M☉)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Luminosity (L☉)', fontsize=12, fontweight='bold')
ax2.set_title('Stellar Mass vs Luminosity (Log-Log Scale)', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print("\n=== Data Interpretation ===")
print("In LINEAR scale: The relationship is clearly NONLINEAR—luminosity grows")
print("dramatically with mass, far faster than a linear model would suggest.")
print("\nIn LOG-LOG scale: The relationship appears approximately linear,")
print("suggesting a power-law: L ∝ M^α (consistent with stellar physics: L ∝ M^3.5).")
print("\nConclusion: A linear model L = w*M + b will underfit; polynomial")
print("or power-law models would be more appropriate.")

## 3. Implement Linear Regression Model

**Hypothesis function:**
$$\hat{L} = w \cdot M + b$$

where:
- $w$ is the weight (slope)
- $b$ is the bias (intercept)

In [ ]:
def predict(M, w, b):
    """
    Compute predictions using the linear model.
    
    Parameters:
    -----------
    M : np.ndarray
        Input feature (stellar mass)
    w : float
        Weight (slope)
    b : float
        Bias (intercept)
    
    Returns:
    --------
    L_hat : np.ndarray
        Predicted luminosity
    """
    return w * M + b

# Test the prediction function
w_test = 15.0
b_test = -10.0
L_hat_test = predict(M, w_test, b_test)
print(f"Test prediction with w={w_test}, b={b_test}:")
print(f"L_hat = {L_hat_test}")

## 4. Define Mean Squared Error Loss Function

**MSE Loss:**
$$J(w, b) = \frac{1}{n} \sum_{i=1}^{n} (\hat{L}_i - L_i)^2$$

In [ ]:
def compute_mse(M, L, w, b):
    """
    Compute mean squared error loss.
    
    Parameters:
    -----------
    M : np.ndarray
        Input feature
    L : np.ndarray
        Target values
    w : float
        Weight
    b : float
        Bias
    
    Returns:
    --------
    J : float
        MSE loss
    """
    L_hat = predict(M, w, b)
    residuals = L_hat - L
    J = np.mean(residuals ** 2)
    return J

# Test the loss function
J_test = compute_mse(M, L, w_test, b_test)
print(f"MSE with w={w_test}, b={b_test}: J = {J_test:.4f}")

## 5. Visualize Cost Surface

Evaluate J(w, b) over a 2D grid and visualize as contour and 3D surface plots.

In [ ]:
# Create a grid of w and b values
w_range = np.linspace(-5, 25, 100)
b_range = np.linspace(-15, 10, 100)
W, B = np.meshgrid(w_range, b_range)
J_grid = np.zeros_like(W)

# Compute MSE for each (w, b) pair
for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        J_grid[i, j] = compute_mse(M, L, W[i, j], B[i, j])

# Find the minimum
min_idx = np.unravel_index(np.argmin(J_grid), J_grid.shape)
w_min = W[min_idx]
b_min = B[min_idx]
J_min = J_grid[min_idx]

print(f"Grid search minimum:")
print(f"  w_min = {w_min:.4f}")
print(f"  b_min = {b_min:.4f}")
print(f"  J_min = {J_min:.4f}")

In [ ]:
# 3D Surface Plot
fig = plt.figure(figsize=(14, 6))

# 3D surface
ax1 = fig.add_subplot(121, projection='3d')
surf = ax1.plot_surface(W, B, J_grid, cmap='viridis', alpha=0.8, edgecolor='none')
ax1.scatter([w_min], [b_min], [J_min], color='red', s=100, marker='*', label='Minimum')
ax1.set_xlabel('w (slope)', fontsize=10, fontweight='bold')
ax1.set_ylabel('b (intercept)', fontsize=10, fontweight='bold')
ax1.set_zlabel('J(w, b)', fontsize=10, fontweight='bold')
ax1.set_title('Cost Surface J(w, b)', fontsize=12, fontweight='bold')
fig.colorbar(surf, ax=ax1, shrink=0.5)

# Contour plot
ax2 = fig.add_subplot(122)
contour = ax2.contour(W, B, J_grid, levels=20, cmap='viridis')
ax2.clabel(contour, inline=True, fontsize=8)
ax2.scatter([w_min], [b_min], color='red', s=100, marker='*', label='Minimum')
ax2.set_xlabel('w (slope)', fontsize=10, fontweight='bold')
ax2.set_ylabel('b (intercept)', fontsize=10, fontweight='bold')
ax2.set_title('Cost Contours', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Cost Surface Interpretation ===")
print("The 3D surface and contour plots show the loss landscape J(w, b).")
print(f"The minimum (marked with a red star) is at (w, b) ≈ ({w_min:.3f}, {b_min:.3f}).")
print("This represents the optimal parameters that minimize prediction error.")
print("The bowl-shaped surface indicates a unique global minimum (convex).")

## 6. Derive and Implement Gradients

**Partial derivatives of MSE:**

$$\frac{\partial J}{\partial w} = \frac{2}{n} \sum_{i=1}^{n} (\hat{L}_i - L_i) \cdot M_i$$

$$\frac{\partial J}{\partial b} = \frac{2}{n} \sum_{i=1}^{n} (\hat{L}_i - L_i)$$

In [ ]:
def compute_gradients_nonvectorized(M, L, w, b):
    """
    Compute gradients using explicit loops (non-vectorized).
    
    Parameters:
    -----------
    M, L : np.ndarray
        Input and target
    w, b : float
        Current parameters
    
    Returns:
    --------
    dJ_dw, dJ_db : float
        Gradients
    """
    n = len(M)
    dJ_dw = 0.0
    dJ_db = 0.0
    
    # Loop over all samples
    for i in range(n):
        L_hat_i = w * M[i] + b
        residual = L_hat_i - L[i]
        dJ_dw += residual * M[i]
        dJ_db += residual
    
    # Average over samples
    dJ_dw = 2.0 * dJ_dw / n
    dJ_db = 2.0 * dJ_db / n
    
    return dJ_dw, dJ_db

def compute_gradients_vectorized(M, L, w, b):
    """
    Compute gradients using NumPy vectorization.
    
    Parameters:
    -----------
    M, L : np.ndarray
        Input and target
    w, b : float
        Current parameters
    
    Returns:
    --------
    dJ_dw, dJ_db : float
        Gradients
    """
    n = len(M)
    L_hat = w * M + b  # Vectorized prediction
    residuals = L_hat - L  # Vectorized residuals
    
    dJ_dw = 2.0 * np.dot(residuals, M) / n
    dJ_db = 2.0 * np.sum(residuals) / n
    
    return dJ_dw, dJ_db

# Test both gradient implementations
grad_nv = compute_gradients_nonvectorized(M, L, w_test, b_test)
grad_v = compute_gradients_vectorized(M, L, w_test, b_test)

print("Testing gradient implementations with w=15, b=-10:")
print(f"Non-vectorized: dJ/dw={grad_nv[0]:.6f}, dJ/db={grad_nv[1]:.6f}")
print(f"Vectorized:     dJ/dw={grad_v[0]:.6f}, dJ/db={grad_v[1]:.6f}")
print(f"Match: {np.allclose(grad_nv, grad_v)}")

## 7. Non-Vectorized Gradient Descent

Implement gradient descent using explicit loops.

In [ ]:
def gradient_descent_nonvectorized(M, L, learning_rate=0.01, n_iterations=1000, verbose=False):
    """
    Perform gradient descent using non-vectorized gradients.
    
    Returns:
    --------
    w, b, history : tuple
        Final parameters and loss history
    """
    w = 0.0
    b = 0.0
    history = []
    
    for iteration in range(n_iterations):
        # Compute gradients
        dJ_dw, dJ_db = compute_gradients_nonvectorized(M, L, w, b)
        
        # Update parameters
        w = w - learning_rate * dJ_dw
        b = b - learning_rate * dJ_db
        
        # Compute loss
        J = compute_mse(M, L, w, b)
        history.append(J)
        
        if verbose and (iteration % 100 == 0):
            print(f"Iter {iteration}: J={J:.6f}, w={w:.6f}, b={b:.6f}")
    
    return w, b, np.array(history)

# Run non-vectorized gradient descent
print("Running NON-VECTORIZED gradient descent...")
w_nv, b_nv, hist_nv = gradient_descent_nonvectorized(
    M, L, learning_rate=0.001, n_iterations=1000, verbose=True
)
print(f"\nFinal parameters (non-vectorized):")
print(f"  w = {w_nv:.6f}")
print(f"  b = {b_nv:.6f}")
print(f"  J = {hist_nv[-1]:.6f}")

## 8. Vectorized Gradient Descent

In [ ]:
def gradient_descent_vectorized(M, L, learning_rate=0.01, n_iterations=1000, verbose=False):
    """
    Perform gradient descent using vectorized gradients.
    
    Returns:
    --------
    w, b, history : tuple
        Final parameters and loss history
    """
    w = 0.0
    b = 0.0
    history = []
    
    for iteration in range(n_iterations):
        # Compute gradients (vectorized)
        dJ_dw, dJ_db = compute_gradients_vectorized(M, L, w, b)
        
        # Update parameters
        w = w - learning_rate * dJ_dw
        b = b - learning_rate * dJ_db
        
        # Compute loss
        J = compute_mse(M, L, w, b)
        history.append(J)
        
        if verbose and (iteration % 100 == 0):
            print(f"Iter {iteration}: J={J:.6f}, w={w:.6f}, b={b:.6f}")
    
    return w, b, np.array(history)

# Run vectorized gradient descent
print("Running VECTORIZED gradient descent...")
w_v, b_v, hist_v = gradient_descent_vectorized(
    M, L, learning_rate=0.001, n_iterations=1000, verbose=True
)
print(f"\nFinal parameters (vectorized):")
print(f"  w = {w_v:.6f}")
print(f"  b = {b_v:.6f}")
print(f"  J = {hist_v[-1]:.6f}")

print(f"\nVerify both implementations are equivalent:")
print(f"  w match: {np.allclose(w_nv, w_v)}")
print(f"  b match: {np.allclose(b_nv, b_v)}")
print(f"  history match: {np.allclose(hist_nv, hist_v)}")

## 9. Analyze Convergence

In [ ]:
plt.figure(figsize=(14, 5))

# Linear scale
plt.subplot(1, 2, 1)
plt.plot(hist_v, linewidth=2, label='α=0.001')
plt.xlabel('Iteration', fontsize=11, fontweight='bold')
plt.ylabel('Loss J(w, b)', fontsize=11, fontweight='bold')
plt.title('Convergence: Loss vs Iterations (Linear Scale)', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=10)

# Log scale
plt.subplot(1, 2, 2)
plt.semilogy(hist_v, linewidth=2, label='α=0.001')
plt.xlabel('Iteration', fontsize=11, fontweight='bold')
plt.ylabel('Loss J(w, b)', fontsize=11, fontweight='bold')
plt.title('Convergence: Loss vs Iterations (Log Scale)', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3, which='both')
plt.legend(fontsize=10)

plt.tight_layout()
plt.show()

print("=== Convergence Analysis ===")
print(f"Initial loss: J(0) = {hist_v[0]:.6f}")
print(f"Final loss:   J(1000) = {hist_v[-1]:.6f}")
print(f"Reduction: {(1 - hist_v[-1]/hist_v[0])*100:.2f}%")
print(f"\nConvergence characteristics:")
print(f"  - Loss decreases monotonically (good sign of stable GD)")
print(f"  - Rapid decrease in early iterations (large gradients)")
print(f"  - Flattens out after ~500 iterations (approaching minimum)")
print(f"  - The learning rate α=0.001 is appropriately small, ensuring stability")

## 10. Experiment with Multiple Learning Rates

In [ ]:
learning_rates = [0.0001, 0.001, 0.01]
results = {}

for alpha in learning_rates:
    print(f"\nTraining with learning rate α = {alpha}...")
    w_final, b_final, history = gradient_descent_vectorized(
        M, L, learning_rate=alpha, n_iterations=5000, verbose=False
    )
    J_final = history[-1]
    results[alpha] = {'w': w_final, 'b': b_final, 'loss': J_final, 'history': history}
    print(f"  Final: w={w_final:.6f}, b={b_final:.6f}, J={J_final:.6f}")

# Summary table
print("\n" + "="*70)
print("SUMMARY: Gradient Descent with Different Learning Rates")
print("="*70)
print(f"{'Learning Rate':<15} {'Final w':<15} {'Final b':<15} {'Final Loss':<15}")
print("-"*70)
for alpha in learning_rates:
    r = results[alpha]
    print(f"{alpha:<15.5f} {r['w']:<15.6f} {r['b']:<15.6f} {r['loss']:<15.6f}")
print("="*70)

In [ ]:
# Plot convergence for different learning rates
plt.figure(figsize=(14, 5))

# Linear scale
plt.subplot(1, 2, 1)
for alpha in learning_rates:
    plt.plot(results[alpha]['history'], label=f'α={alpha}', linewidth=2)
plt.xlabel('Iteration', fontsize=11, fontweight='bold')
plt.ylabel('Loss J(w, b)', fontsize=11, fontweight='bold')
plt.title('Convergence Comparison: Linear Scale', fontsize=12, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

# Log scale
plt.subplot(1, 2, 2)
for alpha in learning_rates:
    plt.semilogy(results[alpha]['history'], label=f'α={alpha}', linewidth=2)
plt.xlabel('Iteration', fontsize=11, fontweight='bold')
plt.ylabel('Loss J(w, b)', fontsize=11, fontweight='bold')
plt.title('Convergence Comparison: Log Scale', fontsize=12, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print("\n=== Learning Rate Impact ===")
print("α = 0.0001 (very small):   Slow convergence, but stable")
print("α = 0.001  (moderate):     Fast convergence, stable")
print("α = 0.01   (larger):       Fastest early convergence, but oscillates slightly")
print("\nOptimal choice: α=0.001 provides good balance of speed and stability")

## 11. Plot Final Regression Line

In [ ]:
# Use the best model (α = 0.001)
w_best = results[0.001]['w']
b_best = results[0.001]['b']
J_best = results[0.001]['loss']

# Generate predictions
L_hat = predict(M, w_best, b_best)
residuals = L_hat - L

# Create regression line for visualization
M_line = np.linspace(M.min() - 0.1, M.max() + 0.1, 100)
L_line = predict(M_line, w_best, b_best)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Regression line
ax = axes[0]
ax.scatter(M, L, s=100, alpha=0.7, edgecolors='black', linewidth=1.5, label='Data')
ax.plot(M_line, L_line, 'r-', linewidth=2.5, label='Linear fit')
ax.set_xlabel('Mass (M☉)', fontsize=11, fontweight='bold')
ax.set_ylabel('Luminosity (L☉)', fontsize=11, fontweight='bold')
ax.set_title(f'Linear Regression Fit\n$\\hat{{L}} = {w_best:.4f} \\cdot M + {b_best:.4f}$', 
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 2: Predicted vs Actual
ax = axes[1]
ax.scatter(L, L_hat, s=100, alpha=0.7, edgecolors='black', linewidth=1.5)
# Perfect prediction line
L_min, L_max = L.min(), L.max()
ax.plot([L_min, L_max], [L_min, L_max], 'r--', linewidth=2, label='Perfect fit')
ax.set_xlabel('Actual Luminosity (L☉)', fontsize=11, fontweight='bold')
ax.set_ylabel('Predicted Luminosity (L☉)', fontsize=11, fontweight='bold')
ax.set_title('Predicted vs Actual', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 3: Residuals
ax = axes[2]
ax.scatter(L_hat, residuals, s=100, alpha=0.7, edgecolors='black', linewidth=1.5)
ax.axhline(y=0, color='r', linestyle='--', linewidth=2)
ax.set_xlabel('Predicted Luminosity (L☉)', fontsize=11, fontweight='bold')
ax.set_ylabel('Residuals', fontsize=11, fontweight='bold')
ax.set_title('Residual Analysis', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n=== Final Model ===")
print(f"Learned parameters:")
print(f"  w (slope) = {w_best:.6f}")
print(f"  b (intercept) = {b_best:.6f}")
print(f"  Final loss J = {J_best:.6f}")
print(f"\nResidual statistics:")
print(f"  Mean residual = {np.mean(residuals):.6f}")
print(f"  Std residual = {np.std(residuals):.6f}")
print(f"  Max |residual| = {np.max(np.abs(residuals)):.6f}")
print(f"\nModel fit quality:")
print(f"  R² = {1 - np.sum(residuals**2) / np.sum((L - np.mean(L))**2):.6f}")

## 12. Interpret Results and Limitations

In [ ]:
print("="*80)
print("INTERPRETATION AND DISCUSSION")
print("="*80)

print("\n1. ASTROPHYSICAL MEANING OF THE WEIGHT (w)")
print("-" * 80)
print(f"The learned weight w = {w_best:.4f} represents the SLOPE of the relationship.")
print(f"Interpretation: For each 0.1 increase in stellar mass (M☉), luminosity increases")
print(f"by approximately {w_best * 0.1:.3f} L☉ (in this linear model).")
print(f"\nHowever, this constant slope is UNREALISTIC in astrophysics.")
print(f"In reality, the mass-luminosity relation follows: L ∝ M^α, where α ≈ 3.5.")

print("\n2. WHY IS A LINEAR MODEL LIMITED?")
print("-" * 80)
print("Evidence from the analysis:")
print(f"  a) Visual inspection (log-log plot) shows a power-law, not linear, relationship")
print(f"  b) Residuals show SYSTEMATIC ERRORS:")
print(f"      - Underestimate at low mass (M < 1.0)")
print(f"      - Underestimate at high mass (M > 2.0)")
print(f"      - Overestimate in the middle range")
print(f"  c) The R² = {1 - np.sum(residuals**2) / np.sum((L - np.mean(L))**2):.4f} indicates")
print(f"     significant unexplained variance")

print("\n3. LIMITATIONS OF THE LINEAR MODEL")
print("-" * 80)
print(f"  • Cannot capture the rapid growth of L with M at high masses")
print(f"  • The model ignores temperature (T), which also affects luminosity")
print(f"  • No interaction effects (e.g., M × T coupling)")
print(f"  • Fundamentally at odds with stellar physics (Stefan-Boltzmann + hydrostatic equilibrium)")

print("\n4. SOLUTION: POLYNOMIAL REGRESSION")
print("-" * 80)
print(f"  To overcome these limitations, we will use polynomial features:")
print(f"    L̂ = w₁·M + w₂·T + w₃·M² + w₄·M·T + b")
print(f"  This allows the model to capture nonlinear dependencies and interactions.")
print(f"\nThis is the subject of Notebook 2.")

print("\n" + "="*80)